In [1]:
import os
import torch
import gc 

from ultralytics import YOLO, RTDETR

import types

In [2]:
# ==================== КОНФИГУРАЦИЯ ====================
AUGMENTED_DATASETS_DIR = 'dataset_folds'
NUM_FOLDS = 3

# Количество эпох обучения
EPOCHS_YOLO = 100     
EPOCHS_DETR = 130    
 
# Параметры обучения
BATCH_SIZE = 8         
NUM_CLASSES = 2       
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Конфигурация: {NUM_FOLDS} фолдов, YOLO={EPOCHS_YOLO} эпох, RT-DETR={EPOCHS_DETR} эпох")
print(f"Устройство: {DEVICE}")
print(f"Датасеты: {AUGMENTED_DATASETS_DIR}/fold_N_augmented/")

Конфигурация: 3 фолдов, YOLO=100 эпох, RT-DETR=130 эпох
Устройство: cuda
Датасеты: dataset_folds/fold_N_augmented/


In [3]:
def free_memory():
    """Агрессивная очистка RAM и VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def on_epoch_end_callback(*args, **kwargs):
    """Коллбек для Ultralytics, вызывается в конце каждой эпохи."""
    free_memory()

In [4]:
# ==================== YOLO11M TRAINING ====================
def train_yolo11_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"\n{'='*60}")
    print(f" Обучение YOLO11m | Фолд {fold_idx} | 100 эпох | imgsz=1024")
    print(f"{'='*60}\n")
    
    yolo_model = YOLO('yolo11m.pt')
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    
    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=768,
        patience=10,
        project='runs/ensemble_training',
        name=f'yolo11m_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        # Loss weights
        box=7.5,
        cls=1.0,
        dfl=1.5,
        
        # Optimizer
        cos_lr=True,
        
        # Геометрические аугментации (усиленные)
        fliplr=0.5,
        flipud=0.0,
        degrees=35.0,        # Увеличено с 30 до 45
        translate=0.25,      # Увеличено с 0.2 до 0.25
        scale=0.3,           # Увеличено с 0.6 до 0.7
        shear=15.0,          # Увеличено с 20 до 25
        perspective=0.0001,
        
        # Цветовые аугментации
        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,
        
        # Advanced аугментации
        mosaic=0.6,          # Увеличено с 0.5 до 0.8
        mixup=0.25,          # Увеличено с 0.15 до 0.25
        cutmix=0.15,         # Добавлено (было 0.0)
        copy_paste=0.3,      # Увеличено с 0.3 до 0.4
        erasing=0.4,
        
        close_mosaic=15,     # Увеличено для стабильности
    )
    
    del yolo_model
    free_memory()
    print(f" YOLO11m Фолд {fold_idx} завершён\n")

In [5]:
# ==================== YOLO26M TRAINING ====================
def train_yolo26_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"\n{'='*60}")
    print(f" Обучение YOLO26m | Фолд {fold_idx} | 100 эпох | imgsz=1024")
    print(f"{'='*60}\n")
    
    yolo_model = YOLO('yolo26m.pt')
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    
    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=768,
        patience=10,
        project='runs/ensemble_training',
        name=f'yolo26m_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        # Loss weights
        box=7.5,
        cls=1.0,
        dfl=1.5,
        
        # Optimizer
        cos_lr=True,
        
        # Геометрические аугментации (усиленные)
        fliplr=0.5,
        flipud=0.0,
        degrees=35.0,
        translate=0.25,
        scale=0.3,
        shear=15.0,
        perspective=0.0001,
        
        # Цветовые аугментации
        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,
        
        # Advanced аугментации
        mosaic=0.6,
        mixup=0.25,
        cutmix=0.15,
        copy_paste=0.3,
        erasing=0.4,
        
        close_mosaic=15,
    )
    
    del yolo_model
    free_memory()
    print(f" YOLO26m Фолд {fold_idx} завершён\n")

In [6]:
# ==================== RT-DETR TRAINING ====================
def custom_optimizer_step(self):
    """Кастомный шаг оптимизатора с жестким клиппингом градиентов"""
    self.scaler.unscale_(self.optimizer)
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    self.scaler.step(self.optimizer)
    self.scaler.update()
    self.optimizer.zero_grad()
    if self.ema:
        self.ema.update(self.model)

def on_train_start_add_clip(trainer):
    trainer.optimizer_step = types.MethodType(custom_optimizer_step, trainer)
    print(" Gradient Clipping (max_norm=1.0) активирован!")

def train_rtdetrv2_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"\n{'='*60}")
    print(f" Обучение RT-DETR | Фолд {fold_idx} | 150 эпох | imgsz=1024")
    print(f"{'='*60}\n")
    
    rtdetr_model = RTDETR('rtdetr-l.pt')
    rtdetr_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    rtdetr_model.add_callback("on_train_start", on_train_start_add_clip)

    rtdetr_model.train(
        data=yaml_path,
        epochs=EPOCHS_DETR,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,           # Больше patience для DETR
        project='runs/ensemble_training',
        name=f'rtdetr_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        amp=True,
        save_period=10,
        pretrained=True,

        # Loss weights
        box=5.0,
        cls=1.0,
        dfl=1.5,

        # Optimizer
        optimizer='AdamW',
        lr0=0.0001,
        cos_lr=True,

        # Геометрические аугментации
        fliplr=0.5,
        flipud=0.0,
        degrees=35.0,
        translate=0.25,
        scale=0.3,
        shear=15.0,
        perspective=0.0001,
        
        # Цветовые аугментации
        hsv_h=0.02,
        hsv_s=0.35,
        hsv_v=0.35,
        bgr=0.0,

        # Advanced аугментации
        mosaic=0.6,
        mixup=0.25,
        cutmix=0.15,
        copy_paste=0.3,
        erasing=0.4,
        
        close_mosaic=15,
    )
    
    del rtdetr_model
    free_memory()
    print(f" RT-DETR Фолд {fold_idx} завершён\n")

In [ ]:
# ==================== MAIN TRAINING LOOP ====================
def main():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    free_memory()
    
    print("\n" + "="*60)
    print(f" ЗАПУСК ОБУЧЕНИЯ | {NUM_FOLDS} ФОЛДА | 3 МОДЕЛИ")
    print("="*60)
    print(f"\nМодели для обучения:")
    print(f"  1. YOLO11m  - {EPOCHS_YOLO} эпох, imgsz=768")
    print(f"  2. YOLO26m  - {EPOCHS_YOLO} эпох, imgsz=768")
    print(f"  3. RT-DETRv2 - {EPOCHS_DETR} эпох, imgsz=640")
    print(f"\nВсего моделей: {NUM_FOLDS * 3}")
    print("="*60 + "\n")
    
    for fold_idx in range(1, NUM_FOLDS + 1):
        fold_dir = os.path.join(AUGMENTED_DATASETS_DIR, f'fold_{fold_idx}')
        
        print(f"\n{'='*60}")
        print(f" ФОЛД {fold_idx}/{NUM_FOLDS}")
        print(f"{'='*60}\n")
        
        # Обучение всех 3 моделей на каждом фолде
        if fold_idx != 1:
            train_yolo11_model(fold_idx, fold_dir)
            train_yolo26_model(fold_idx, fold_dir)
        train_rtdetrv2_model(fold_idx, fold_dir)
        
        print(f"\n Фолд {fold_idx} завершён. Очистка памяти...\n")
        free_memory()
    
    print("\n" + "="*60)
    print("  ОБУЧЕНИЕ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНО! ")
    print("="*60)
    print(f"\nСохранённые модели:")
    print(f"  runs/ensemble_training/yolo11m_fold_*/weights/best.pt")
    print(f"  runs/ensemble_training/yolo26m_fold_*/weights/best.pt")
    print(f"  runs/ensemble_training/rtdetrv2_fold_*/weights/best.pt")
    print(f"\nВсего: {NUM_FOLDS * 3} моделей")
    print("="*60 + "\n")

In [ ]:
# Запуск обучения
if __name__ == "__main__":
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    main()